# MOSTA mouse organogenesis

This notebook is a guide to the `mosta` workflow. It shows how raw
data are mapped into the fields expected by CytoBridge, how a new model run is
passed to downstream analysis, and which later steps start from files saved
for the paper instead of the new run. Edit the paths in **Setup** before
starting.

For a small example that runs on generated data, see [Synthetic
preprocessing](../data_preparation/synthetic_preprocessing.ipynb). The [data and
checkpoint guide](../../data_checkpoints.md) lists inputs distributed outside
the package.

## Setup

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from CytoBridge.workflow import (
    WorkflowOptions,
    build_workflow_plan,
    load_workflow_config,
    render_workflow_plan,
    run_workflow,
)
DATASET_CONFIG = 'mosta'
RAW_H5AD = Path("data/mosta_raw.h5ad")
OUTPUT_DIR = Path("tutorial_outputs/mosta")
PREPROCESS_ONLY_DIR = Path("tutorial_outputs/mosta_preprocess_only")
DOWNSTREAM_RERUN_DIR = Path("tutorial_outputs/mosta_downstream_rerun")
ALIGNED_H5AD = OUTPUT_DIR / "preprocess" / 'mosta_aligned.h5ad'
MODEL_DIR = OUTPUT_DIR / "training"


RUN_TRAINING = False
RUN_PREPROCESS_ONLY = False
RUN_DOWNSTREAM = False

In [2]:
config, config_source = load_workflow_config(DATASET_CONFIG)
dataset = config["dataset"]
scientific = config["scientific"]
downstream = config["downstream"]
preprocess = config["preprocess"]
align = preprocess["align"]

spatial_obs_keys = align.get("spatial_obs_keys")
if spatial_obs_keys:
    spatial_source = ", ".join(f"obs[{key!r}]" for key in spatial_obs_keys)
else:
    spatial_source = f"obsm[{align.get('input_spatial_key', 'spatial')!r}]"

pd.DataFrame(
    {
        "setting": [
            "dataset",
            "configuration",
            "raw time column",
            "raw annotation column",
            "aligned annotation column",
            "raw count layer",
            "raw spatial coordinates",
            "aligned spatial coordinates",
            "model time values",
            "classifier neighbors",
        ],
        "value": [
            dataset["display_name"],
            config_source,
            preprocess["time_key"],
            preprocess["annotation_source"],
            dataset["annotation_key"],
            align.get("expression_layer", "X"),
            spatial_source,
            f"obsm[{dataset['spatial_key']!r}]",
            ", ".join(map(str, align["time_mapping"].values())),
            scientific["classifier_k"],
        ],
    }
)

,setting,value
0,dataset,MOSTA mouse organogenesis
1,configuration,example configuration: mosta
2,raw time column,timepoint
3,raw annotation column,annotation
4,aligned annotation column,Annotation
5,raw count layer,count
6,raw spatial coordinates,obsm['spatial']
7,aligned spatial coordinates,obsm['spatial_aligned']
8,model time values,"-3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0, 4.0"
9,classifier neighbors,10


## Start a new model run

The dataset configuration records the count layer, time mapping, spatial
coordinates, alignment settings, and model settings. Start here when fitting a
new model. The command reads the raw H5AD, writes the aligned H5AD, fits the
ligand--receptor edge predictor when the model uses one, trains CytoBridge, and
runs the analyses selected in the configuration. No separate preprocessing
command is needed first.

### Run a new dataset from raw counts

```bash
cytobridge workflow --config mosta --train \
  --input-h5ad <raw.h5ad> --output-dir <run> --device cuda
```

**Input:** raw H5AD, dataset configuration, and the included or user-supplied LR database

**Output:** `<run>`/preprocess/mosta_aligned.h5ad; `<run>`/preprocess/edge_classifier/mosta_edge_model.pt when the configuration uses a ligand--receptor edge predictor; `<run>`/training/`<stage>`/best_model.pth or score_model.pth; `<run>`/training/adata.h5ad; `<run>`/training/training_history.csv; `<run>`/training/training_run_summary.json; `<run>`/downstream/summary.json, result tables, and figures

**Continue with:** inspect `<run>`/downstream, or rerun only the downstream analysis in a new output directory

Start here with raw data. This one command preprocesses, trains, and runs the analyses selected in the configuration. The separate preprocessing-only run below is optional.

The next two cells show the same operation through the Python API. Leave
`RUN_TRAINING = False` when reading the documentation; set it to `True` only
after the paths above point to your data. The compact table shows the file
used by each step; the complete package plan is stored in `training_plan_text` if you
want to print it in Jupyter.

In [3]:
training_options = WorkflowOptions(
    input_h5ad=RAW_H5AD,
    output_dir=OUTPUT_DIR,
    train=True,
)
training_plan = build_workflow_plan(
    config,
    source=config_source,
    options=training_options,
)
training_plan_text = render_workflow_plan(training_plan)
pd.DataFrame(
    [
        {
            "step": "preprocess",
            "input": RAW_H5AD,
            "output": ALIGNED_H5AD,
        },
        {
            "step": "train",
            "input": ALIGNED_H5AD,
            "output": MODEL_DIR,
        },
        {
            "step": "downstream",
            "input": f"{ALIGNED_H5AD} + {MODEL_DIR}",
            "output": OUTPUT_DIR / "downstream",
        },
    ]
)

,step,input,output
0,preprocess,data/mosta_raw.h5ad,tutorial_outputs/mosta/preprocess/mosta_aligne...
1,train,tutorial_outputs/mosta/preprocess/mosta_aligne...,tutorial_outputs/mosta/training
2,downstream,tutorial_outputs/mosta/preprocess/mosta_aligne...,tutorial_outputs/mosta/downstream


In [4]:
if RUN_TRAINING:
    if not RAW_H5AD.is_file():
        raise FileNotFoundError(f"Update RAW_H5AD before training: {RAW_H5AD}")
    training_result = run_workflow(config, options=training_options)
    training_result
else:
    print("Training is off. Set RUN_TRAINING = True to start a new model run.")

Training is off. Set RUN_TRAINING = True to start a new model run.


## Inspect the aligned data without training (optional)

Use this separate command only when you want to examine the aligned H5AD before
committing to a model fit. It writes to `PREPROCESS_ONLY_DIR` and does not fit
an edge predictor or a CytoBridge model. It is not an earlier step in the model
run above; when you are ready to train, use the first command from the raw H5AD.

```bash
cytobridge workflow --config mosta --step preprocess \
  --input-h5ad <raw.h5ad> --output-dir <preprocess-only>
```

**Input:** raw H5AD and the dataset configuration

**Output:** `<preprocess-only>`/preprocess/mosta_aligned.h5ad and preprocessing records; no edge predictor or CytoBridge model

**Continue with:** inspect the aligned H5AD; use the first command, starting again from the raw H5AD, when ready to fit a model

This is an alternative inspection run, not a prerequisite for training.

In [5]:
preprocess_only_options = WorkflowOptions(
    input_h5ad=RAW_H5AD,
    output_dir=PREPROCESS_ONLY_DIR,
    steps=("preprocess",),
)
preprocess_only_plan = build_workflow_plan(
    config,
    source=config_source,
    options=preprocess_only_options,
)
preprocess_only_plan_text = render_workflow_plan(preprocess_only_plan)
pd.DataFrame(
    [
        {
            "step": "preprocess only",
            "input": RAW_H5AD,
            "output": PREPROCESS_ONLY_DIR / "preprocess" / ALIGNED_H5AD.name,
        }
    ]
)

,step,input,output
0,preprocess only,data/mosta_raw.h5ad,tutorial_outputs/mosta_preprocess_only/preproc...


In [6]:
if RUN_PREPROCESS_ONLY:
    if not RAW_H5AD.is_file():
        raise FileNotFoundError(f"Update RAW_H5AD before preprocessing: {RAW_H5AD}")
    preprocess_only_result = run_workflow(
        config,
        options=preprocess_only_options,
    )
    preprocess_only_result
else:
    print(
        "Preprocessing-only run is off. Set RUN_PREPROCESS_ONLY = True "
        "to write an aligned H5AD without training."
    )

Preprocessing-only run is off. Set RUN_PREPROCESS_ONLY = True to write an aligned H5AD without training.


## Run downstream analysis again (optional)

The first command already runs downstream analysis. Use this section only when
you want to repeat it from the same aligned H5AD and fitted model. The rerun
writes to `DOWNSTREAM_RERUN_DIR`, leaving the original results unchanged.

```bash
cytobridge workflow --config mosta --step downstream \
  --aligned-h5ad <run>/preprocess/mosta_aligned.h5ad \
  --model-dir <run>/training --output-dir <downstream-rerun>
```

**Input:** aligned H5AD; `<run>`/training; dataset-matched LR database

**Output:** `<downstream-rerun>`/downstream/summary.json; slice_data/*.h5ad; velocity/velocity_components.npz; growth/growth_by_cell.csv; composition/celltype_composition.csv; communication and ligand_receptor tables; standard figures

**Continue with:** paper-specific continuation shown in the paper-figure notebook

The first command already runs these analyses. Use this form only to repeat downstream analysis, and choose a new output directory so the original results are not overwritten.

In [7]:
downstream_options = WorkflowOptions(
    aligned_h5ad=ALIGNED_H5AD,
    model_dir=MODEL_DIR,
    output_dir=DOWNSTREAM_RERUN_DIR,
    steps=("downstream",),
)
downstream_plan = build_workflow_plan(
    config,
    source=config_source,
    options=downstream_options,
)
downstream_plan_text = render_workflow_plan(downstream_plan)
pd.DataFrame(
    [
        {
            "step": "downstream",
            "input": f"{ALIGNED_H5AD}; {MODEL_DIR}",
            "output": DOWNSTREAM_RERUN_DIR / "downstream",
        }
    ]
)

,step,input,output
0,downstream,tutorial_outputs/mosta/preprocess/mosta_aligne...,tutorial_outputs/mosta_downstream_rerun/downst...


In [8]:
if RUN_DOWNSTREAM:
    missing = [path for path in (ALIGNED_H5AD, MODEL_DIR) if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing aligned data or model directory: {missing}")
    downstream_result = run_workflow(config, options=downstream_options)
    downstream_result
else:
    print("Downstream rerun is off. Set RUN_DOWNSTREAM = True to repeat it.")

Downstream rerun is off. Set RUN_DOWNSTREAM = True to repeat it.


## Paper figures

The headings state exactly where each calculation starts:

- **Continue from the model run above** reads the `OUTPUT_DIR` created here.
- **Start from the paper's saved files** reads the tables, arrays, or models
  retained from the exact paper analysis. It does not read the current
  `OUTPUT_DIR` unless the step says so.
- **Required paper files not included** names an input or page builder that is
  not shipped in this repository; no command is shown in its place.

Commands beginning with `python scripts/...` or `python -m scripts...` must be
run from the root of a cloned source repository. Each step names its input,
output, and the notebook that continues from it.

- [Main Figure 4](../paper_figures/main_figure_4.ipynb)
- [Supplementary Figures S11–S18](../paper_figures/mosta_figures.ipynb)

### Use the MOSTA outputs from the model run above (Main Figure 4; S11-S18)

File or directory to use (not a command): `<run>/downstream`

**Input:** the downstream directory written by the complete raw-data run

**Output:** generated states; growth; composition; lineage when requested; gene-dynamics and LR tables when their inputs are supplied; standard figures already stored under `<run>`/downstream

**Continue with:** inspect `<run>`/downstream, or write a new plotting script for the question being studied

No second downstream command is required. To repeat the downstream analysis, use the optional rerun command above with `<downstream-rerun>` as a new output directory. The standard downstream analysis does not calculate GO-enrichment tables or assemble the paper pages.

### Start from the paper's saved files: export Main Figure 4 and S11-S18 (Main Figure 4; S11-S18)

```bash
python scripts/results/export_mosta_figures.py \
  --release-dir release_artifacts/mosta_package_native_corrected_20260826_v1 \
  --output-dir <figure-dir>
```

**Input:** included MOSTA result release and its recorded panel builders

**Output:** five Main Figure 4 vector panels and eight SI vector pages

**Continue with:** view the exports in the Main Figure 4 and MOSTA paper notebooks

This command redraws the paper's saved result files. It does not convert an arbitrary new downstream directory into the manuscript pages.

## Saved files

- Aligned data: `tutorial_outputs/mosta/preprocess/mosta_aligned.h5ad`
- Training directory: `tutorial_outputs/mosta/training`
- Downstream directory: `tutorial_outputs/mosta/downstream`